# Dhaka Smart Waste V4.0 — One-Click Paper/Reviewer Validation

Run **Runtime → Run all**. No manual cell rerun should be required.

This notebook validates the released Parquet package (43.8M rows), writes paper-ready validation tables/figures, captures the exact runtime environment, and creates release checksums/manifests. The dataset is synthetic; these checks establish internal consistency and reproducibility, not field validation of deployed municipal bins.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil, sys

PROJECT_ROOT = Path('/content/drive/MyDrive/Dhaka Smart Waste V4.0 2025')
CODE_DIR = PROJECT_ROOT / 'CODE_AND_NOTEBOOKS'
OUTPUT_DIR = PROJECT_ROOT / 'VALIDATION_OUTPUTS'
EXTRACT_DIR = Path('/content/dhaka_smart_waste_v4_parquet_validation')
VALIDATOR = CODE_DIR / 'validate_v4_0_paper.py'
REQ = CODE_DIR / 'validation_requirements.txt'

assert PROJECT_ROOT.exists(), f'Missing project folder: {PROJECT_ROOT}'
assert VALIDATOR.exists(), f'Missing validator: {VALIDATOR}'
assert REQ.exists(), f'Missing validation requirements: {REQ}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Remove only artifacts generated by this notebook so stale files cannot be mistaken for a new run.
GENERATED_NAMES = {
    'validation_summary.csv','VALIDATION_REPORT.json','monthly_climate_validation.csv',
    'scenario_summary.csv','parquet_zip_contents.csv','TECHNICAL_VALIDATION_AUTO.md',
    'figure_monthly_rainfall_validation.png','figure_sensor_fill_noise.png',
    'figure_collection_delay_accessibility.png','figure_scenario_effects.png',
    'validator_run.log','environment_lock.txt','environment_summary.txt',
    'RELEASE_FILE_MANIFEST.csv','SHA256SUMS.txt','SUBMISSION_TECHNICAL_CHECKLIST.md',
}
for name in GENERATED_NAMES:
    p = OUTPUT_DIR / name
    if p.exists():
        p.unlink()

print('Project :', PROJECT_ROOT)
print('Code    :', CODE_DIR)
print('Output  :', OUTPUT_DIR)
print('Extract :', EXTRACT_DIR)

## Install validation dependencies and capture the runtime

The exact environment used for this validation is saved after installation so the publication archive has a reproducible software record.

In [ ]:
import subprocess, platform, datetime, json

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQ)], check=True)

import numpy as np, pandas as pd, pyarrow, matplotlib

versions = {
    'python': sys.version.split()[0],
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'pyarrow': pyarrow.__version__,
    'matplotlib': matplotlib.__version__,
    'platform': platform.platform(),
    'machine': platform.machine(),
    'processor': platform.processor(),
    'utc_time': datetime.datetime.now(datetime.timezone.utc).isoformat(),
}

# Exact package lock for this validation run.
freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], text=True, capture_output=True, check=True).stdout
(OUTPUT_DIR / 'environment_lock.txt').write_text(freeze, encoding='utf-8')

free_gb = shutil.disk_usage('/content').free / (1024**3)
summary_lines = [f'{k}: {v}' for k,v in versions.items()] + [f'colab_local_free_gb: {free_gb:.2f}']
(OUTPUT_DIR / 'environment_summary.txt').write_text('\n'.join(summary_lines) + '\n', encoding='utf-8')

for line in summary_lines:
    print(line)

## Pre-flight release check

This fails early if the release layout is incomplete, avoiding a long 43.8M-row scan on a broken package.

In [ ]:
import re, zipfile

zip_dir = PROJECT_ROOT / 'PARQUET_PARTS'
csv_dir = PROJECT_ROOT / 'CSV_PARTS'
zips = sorted(zip_dir.glob('dhaka_smart_waste_v4_0_PARQUET_PART_*_of_23.zip'))
bins_pq = zip_dir / 'bins.parquet'
bins_csv = csv_dir / 'bins.csv.gz'

print('Transport ZIPs :', len(zips))
print('bins.parquet   :', bins_pq.exists())
print('bins.csv.gz    :', bins_csv.exists())
print('Local free GB  :', f'{shutil.disk_usage("/content").free/(1024**3):.2f}')

assert len(zips) == 23, f'Expected 23 Parquet transport ZIPs, found {len(zips)}'
assert bins_pq.exists() or bins_csv.exists(), 'No persistent-bin table found'
assert shutil.disk_usage('/content').free > 4 * 1024**3, 'Less than 4 GiB free local disk; free space before full validation.'

# Fast central-directory check here; the validator performs full CRC testing.
for zp in zips:
    with zipfile.ZipFile(zp, 'r') as zf:
        assert zf.namelist(), f'Empty ZIP: {zp.name}'
print('Pre-flight: PASS')

## Run the complete full-data validator

The validator CRC-checks the 23 ZIP archives, reconstructs/reuses a verified local cache of 92 Parquet parts, scans all 43.8M rows out-of-core, and writes the technical validation outputs. Progress is streamed live and saved to `validator_run.log`.

In [ ]:
cmd = [
    sys.executable,
    str(VALIDATOR),
    '--project-root', str(PROJECT_ROOT),
    '--extract-dir', str(EXTRACT_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--batch-size', '200000',
]
print('Running:', ' '.join(cmd), flush=True)

log_path = OUTPUT_DIR / 'validator_run.log'
with log_path.open('w', encoding='utf-8') as log:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    return_code = proc.wait()

print('\nValidator return code:', return_code)
if return_code != 0:
    raise RuntimeError(f'Validator failed with return code {return_code}. See {log_path}')

required_outputs = [
    'validation_summary.csv','VALIDATION_REPORT.json','monthly_climate_validation.csv',
    'scenario_summary.csv','parquet_zip_contents.csv','TECHNICAL_VALIDATION_AUTO.md'
]
missing = [x for x in required_outputs if not (OUTPUT_DIR/x).exists()]
if missing:
    raise RuntimeError(f'Validator returned success but outputs are missing: {missing}')
print('Validator outputs: PASS')

## Create publication release manifests and SHA256 checksums

This addresses file-integrity/repository reproducibility: every public Drive release file is hashed and recorded, and the ZIP-to-Parquet mapping has already been written by the validator.

In [ ]:
import hashlib

release_files = []
for p in sorted((PROJECT_ROOT/'PARQUET_PARTS').glob('dhaka_smart_waste_v4_0_PARQUET_PART_*_of_23.zip')):
    release_files.append(('parquet_transport_zip', p))
if (PROJECT_ROOT/'PARQUET_PARTS'/'bins.parquet').exists():
    release_files.append(('parquet_static_bins', PROJECT_ROOT/'PARQUET_PARTS'/'bins.parquet'))
for p in sorted((PROJECT_ROOT/'CSV_PARTS').glob('observations_part_*.csv.gz')):
    release_files.append(('csv_observation_part', p))
if (PROJECT_ROOT/'CSV_PARTS'/'bins.csv.gz').exists():
    release_files.append(('csv_static_bins', PROJECT_ROOT/'CSV_PARTS'/'bins.csv.gz'))
if (PROJECT_ROOT/'CSV_PARTS'/'CSV_MANIFEST.csv').exists():
    release_files.append(('csv_manifest', PROJECT_ROOT/'CSV_PARTS'/'CSV_MANIFEST.csv'))

rows = []
for i, (kind, p) in enumerate(release_files, 1):
    h = hashlib.sha256()
    with p.open('rb') as f:
        for chunk in iter(lambda: f.read(8*1024*1024), b''):
            h.update(chunk)
    rel = p.relative_to(PROJECT_ROOT).as_posix()
    digest = h.hexdigest()
    rows.append({'kind':kind, 'relative_path':rel, 'bytes':p.stat().st_size, 'sha256':digest})
    print(f'Hashed {i:02d}/{len(release_files):02d}: {rel}')

manifest = pd.DataFrame(rows)
manifest.to_csv(OUTPUT_DIR/'RELEASE_FILE_MANIFEST.csv', index=False)
(OUTPUT_DIR/'SHA256SUMS.txt').write_text(
    ''.join(f"{r['sha256']}  {r['relative_path']}\n" for r in rows), encoding='utf-8'
)
print('Manifested files:', len(manifest))
print('Total bytes:', f"{manifest['bytes'].sum():,}")

## Master validation table

In [ ]:
from IPython.display import display
summary = pd.read_csv(OUTPUT_DIR / 'validation_summary.csv')
display(summary)

status_counts = summary.groupby(['severity','status']).size().rename('count').reset_index()
print('\nStatus counts:')
display(status_counts)

hard_fail = summary[(summary.severity == 'hard') & (summary.status == 'FAIL')]
soft_warn = summary[(summary.severity == 'soft') & (summary.status == 'WARN')]
print('Hard failures:', len(hard_fail))
print('Soft warnings :', len(soft_warn))
if len(hard_fail):
    display(hard_fail)
if len(soft_warn):
    display(soft_warn)

## Machine-readable report and key measured values

In [ ]:
report = json.loads((OUTPUT_DIR / 'VALIDATION_REPORT.json').read_text(encoding='utf-8'))
m = report['validation_metrics']

keys = [
    'rows_scanned','duplicate_bin_hour_keys','missing_bin_hour_keys',
    'hours_per_bin_min','hours_per_bin_max','packet_loss_rate',
    'sensor_anomaly_rate_available','fill_sensor_noise_mean_pp',
    'fill_sensor_noise_std_pp','forecast_4h_no_collection_mae_pp',
    'forecast_4h_no_collection_n'
]
key_metrics = {k:m.get(k) for k in keys}
display(pd.DataFrame(key_metrics.items(), columns=['metric','value']))
print('Overall hard status:', report.get('overall_hard_status'))

## Climate/calibration and scenario tables

In [ ]:
climate = pd.read_csv(OUTPUT_DIR / 'monthly_climate_validation.csv')
scenario = pd.read_csv(OUTPUT_DIR / 'scenario_summary.csv')
print('Monthly climate validation')
display(climate)
print('\nScenario summaries (simulation implementation checks)')
display(scenario)

## Publication figures

In [ ]:
from IPython.display import Image, display
figs = [
    'figure_monthly_rainfall_validation.png',
    'figure_sensor_fill_noise.png',
    'figure_collection_delay_accessibility.png',
    'figure_scenario_effects.png',
]
for name in figs:
    p = OUTPUT_DIR / name
    if p.exists():
        print('\n', name)
        display(Image(filename=str(p)))
    else:
        print('Not generated:', name)

## Auto-generated Technical Validation text

In [ ]:
tech = (OUTPUT_DIR / 'TECHNICAL_VALIDATION_AUTO.md').read_text(encoding='utf-8')
print(tech)

## Submission technical-readiness gate

This gate covers technical/repository evidence that can be verified automatically. Author declarations and manuscript prose remain manual editorial items.

In [ ]:
hard_fail = summary[(summary.severity == 'hard') & (summary.status == 'FAIL')]
artifacts = [
    'validation_summary.csv','VALIDATION_REPORT.json','monthly_climate_validation.csv',
    'scenario_summary.csv','parquet_zip_contents.csv','RELEASE_FILE_MANIFEST.csv',
    'SHA256SUMS.txt','environment_lock.txt','environment_summary.txt',
    'TECHNICAL_VALIDATION_AUTO.md'
]
missing_artifacts = [x for x in artifacts if not (OUTPUT_DIR/x).exists()]

technical_ready = (len(hard_fail) == 0 and not missing_artifacts)
lines = [
    '# Submission Technical Readiness — Dhaka Smart Waste V4.0',
    '',
    f"- Full release scan: {'PASS' if report.get('validation_metrics',{}).get('rows_scanned') == 43_800_000 else 'FAIL'}",
    f'- Hard validation failures: {len(hard_fail)}',
    f'- Missing technical artifacts: {len(missing_artifacts)}',
    f"- Release SHA256 manifest: {'PASS' if (OUTPUT_DIR/'SHA256SUMS.txt').exists() else 'MISSING'}",
    f"- ZIP-to-Parquet mapping: {'PASS' if (OUTPUT_DIR/'parquet_zip_contents.csv').exists() else 'MISSING'}",
    f"- Exact validation environment captured: {'PASS' if (OUTPUT_DIR/'environment_lock.txt').exists() else 'MISSING'}",
    f"- Technical validation draft: {'PASS' if (OUTPUT_DIR/'TECHNICAL_VALIDATION_AUTO.md').exists() else 'MISSING'}",
    '',
    f"Overall automated technical gate: **{'PASS' if technical_ready else 'FAIL'}**",
    '',
    '## Manual manuscript/repository items still required before journal submission',
    '',
    'These are not invented by the validation notebook and must be completed from author/journal information:',
    '',
    '- Data in Brief manuscript: Abstract, Specifications Table, Value of the Data, Background, Data Description, Experimental Design/Materials and Methods, Limitations.',
    '- Ethics statement appropriate for a fully synthetic dataset.',
    '- CRediT author-contribution statement using the actual author contributions.',
    '- Funding/Acknowledgement statement using the actual funding status.',
    '- Declaration of Competing Interest.',
    '- Final Data Availability and Code Availability statements with DOI/repository links.',
    '- Ensure the public GitHub repository contains a proper root LICENSE file and that canonical generator/validator paths work from a clean clone/tag.',
]
checklist = '\n'.join(lines) + '\n'
(OUTPUT_DIR/'SUBMISSION_TECHNICAL_CHECKLIST.md').write_text(checklist, encoding='utf-8')
print(checklist)

if missing_artifacts:
    raise RuntimeError(f'Missing generated technical artifacts: {missing_artifacts}')
if len(hard_fail):
    raise RuntimeError(f'{len(hard_fail)} hard validation check(s) failed. Do not use results for submission until resolved.')

print('\n✅ ONE-CLICK VALIDATION COMPLETE')
print('✅ Hard failures: 0')
print('✅ Technical/repository artifacts generated')
print('Outputs:', OUTPUT_DIR)